# **Fourier Transformation and Convolution**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.5 <br><br>
<!-- 07.03.2025 / v1.2: Language refactoring, intro only -->
<!-- 10.03.2025 / v1.3: Fixed issue regarding scipy's FFT function -->
<!-- 24.02.2026 / v1.4: Refactored text and introduced overview figure -->
<!-- 09.03.2026 / v1.5: Refactored exercise 1 / 2 / 3 -->
</div>

In this notebook, we will use the FFT to analyze a signal’s frequency content, examine key properties of its spectrum, reconstruct the signal with the inverse FFT, and apply simple frequency‑domain filtering by zeroing selected components.


## **Exercises**
* [Exercise 1 – Familiarize yourself with the FFT](#exercise1)  
* [Exercise 2 – Spectral analysis of audio data](#exercise2)  
* [Exercise 3 – FFT of a periodic signal](#exercise3)  
* [Exercise 4 - Spectrograms](#exercise4)  
* [Exercise 5 – Naive filtering](#exercise5)  

---

## **Preparations**

Let's begin with the usual preparatory steps...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.fft import fft, ifft, fftfreq, fftshift

# For audio playback
from IPython.display import Audio

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Enable this line if you want to use the interactive widgets
# It requires the ipympl package to be installed.
#%matplotlib widget

import sys
sys.path.insert(0, "../")
import isp
# isp.setup_plotting()

---

<a id='exercise1'></a>

## **&#9734;  Exercise 1 – The FFT**

In the following, we will use the FFT functionality provided by `scipy.fft`.  Let's start by getting familiar with the package.


### **Instructions** 

* Read this tutorial: https://realpython.com/python-scipy-fft/  
  (You may skip the sections on discrete cosine and sine transforms.)
* Browse the list of functions in the https://docs.scipy.org/doc/scipy/reference/fft.html module.
* Assume we are given a discrete, real-valued signal `x` with `len(x) == 100`.

* **Answer the following questions**:
  1. What does `fft(x)` return?
  2. What is the meaning of `X[0]` if `X = fft(x)`?
  3. What is the meaning of `X[N//2]` when `X = fft(x)`?
  4. What is the purpose of the functions `fftfreq()` and `fftshift()`?
  5. Which functions are needed to compute an amplitude spectrum and a phase spectrum from a signal `x[t]`?
  6. What is the purpose of the function `ifft()`?
  7. Why is it often useful to use `rfft()` (and `irfft()`)?
  8. What is the role of the argument `n` in `fft(x, n)`?

In [ ]:
######################
###    EXCERISE    ###
######################

# Question 1: ...
# Answer 1: ...

# Question 2: ...
# Answer 2: ...

# Question 3: ...
# Answer 3: ...

# Question 4: ...
# Answer 4: ...

# Question 5: ...
# Answer 5: ...

# Question 6: ...
# Answer 6: ...

# Question 7: ...
# Answer 7: ...

# Question 8: ...
# Answer 8: ...

```python
######################
###    SOLUTION    ###
######################
```

<!-------------------------------- Question 1 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 1:</b>
<span>
What does the function <code>fft(x)</code> return?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 1:</b>
<span>
The function <a href="https://docs.scipy.org/doc/scipy/reference/fft.html" target="_blank" rel="noopener noreferrer"><code>fft(x)</code></a> returns the <strong>Discrete Fourier Transform (DFT)</strong> of the input signal <code>x</code>.
The result is generally a <strong>complex-valued array</strong> of the same length as the input signal.
</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 2 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 2:</b>
<span>
What is the meaning of <code>X[0]</code> if <code>X = fft(x)</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 2:</b>
<span>
Using the definition of the DFT and evaluating it at k=0, we obtain

$$
\begin{aligned}
X[k] &= \sum_{n=0}^{N-1} x[n] e^{-i 2\pi kn/N}\\
X[0] &= \sum_{n=0}^{N-1} x[n].
\end{aligned}
$$

Thus, <code>X[0]</code> equals the <strong>sum of all samples</strong> of the signal. It represents the zero-frequency (DC) component of the spectrum.

Note that the common FFT definition does not normalize the forward DFT (compare with documentation for <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.fft.html#scipy.fft.fft" target="_blank" rel="noopener noreferrer"><code>fft()</code></a>, argument `norm`). Therefore, to obtain the average value of the signal, we must divide <code>X[0]</code> by the number of samples.

$$
\mu(x[n]) = \frac{X[0]}{N}.
$$
</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 3 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 3:</b>
<span>
What is the meaning of <code>X[N//2]</code> if <code>X = fft(x)</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 3:</b>
<span>
The value <code>X[N//2]</code> corresponds to the <strong>Nyquist frequency</strong>. This follows from the fact that the DFT coefficients correspond to the frequencies

$$
f_k = \frac{k}{N} f_s \quad \Rightarrow \quad f_{\tfrac{N}{2}} = \frac{1}{2} f_s.
$$

Thus, <code>X[N//2]</code> represents the spectral component at the Nyquist frequency, which is the highest frequency that can be represented for a signal sampled at frequency $f_s$.
</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 4 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 4:</b>
<span>
What is the purpose of the functions <code>fftfreq()</code> and <code>fftshift()</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 4:</b>
<span>
The function <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.fftfreq.html#scipy.fft.fftfreq" target="_blank" rel="noopener noreferrer"><code>fftfreq()</code></a> returns the <strong>frequency bins</strong> corresponding to the DFT coefficients. In most cases, you would call this function as follows:

```python
    fk = fftfreq(n=len(x), d=1/fs)
```

The function <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.fftshift.html#scipy.fft.fftshift" target="_blank" rel="noopener noreferrer"><code>fftshift()</code></a> shifts the <strong>zero-frequency component to the center</strong> of the spectrum. This is useful for visualization, as it places negative frequencies on the left and positive frequencies on the right.

```python
    # x, fs = ...
    X = fftshift(fft(x))
    fk = fftshift(fftfreq(n=len(x), d=1/fs))
```

</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 5 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 5:</b>
<span>
Which functions can be used to create an amplitude spectrum and a phase spectrum from a signal <code>x[n]</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 5:</b>
<span>We can use <code>np.abs()</code> and <code>np.angle()</code> to compute the magnitude and phase of complex numbers.

```python
    # Almost correct approach:
    # x, fs = ...
    A = np.abs(fft(x))           # Amplitude
    phi = np.angle(fft(x))       # Phase
```

As noted earlier, the forward transform is <strong>not</strong> normalized by default, meaning that the result depends on the length of the signal. It is therefore recommended to normalize the amplitude by dividing by the number of samples. The phase is not affected by this normalization.

Thus, in practice, the spectra are typically computed as follows:

```python
    # Correct approach:
    # x, fs = ...
    A = np.abs(fft(x)) / len(x)  # Normalized amplitude
    phi = np.angle(fft(x))       # Phase
```
</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 6 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 6:</b>
<span>
What is the purpose of the function <code>ifft()</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 6:</b>
<span>
The function <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.ifft.html#scipy.fft.ifft" target="_blank" rel="noopener noreferrer"><code>ifft()</code></a> computes the <strong>inverse Discrete Fourier Transform</strong>. It converts a signal from the frequency domain back to the time domain.

```python
    x_orig = x
    X      = fft(x_orig)   # Forward transform
    x_rec  = ifft(X)       # Inverse transform
    error  = np.max(np.abs(x_orig - x_rec))
    print(error)           # ~0, up to rounding errors
```

</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 7 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 7:</b>
<span>
Why is it often useful to use <code>rfft()</code> (and <code>irfft()</code>)?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 7:</b>
<span>
For <strong>real-valued signals</strong>, the DFT is Hermitian symmetric: the negative-frequency components are the complex conjugates of the positive-frequency components. Therefore, half of the spectrum is redundant.

The function <a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.fft.rfft.html#scipy.fft.rfft" target="_blank" rel="noopener noreferrer"><code>rfft()</code></a> returns only the <strong>non-redundant positive-frequency components</strong>, which reduces both memory usage and computation time. We therefore prefer <code>rfft()</code> over <code>fft()</code> when working with real-valued signals and when only the positive-frequency bins are of interest.

Note that we do not need to reorder the spectrum using <code>fftshift()</code> in this case, because the negative frequencies are not included in the output. However, make sure to use the corresponding function <code>rfftfreq()</code> to obtain the correct frequency bins for the returned spectrum.

```python
    # x, fs = ...
    X = rfft(x)
    fk = rfftfreq(len(x), 1/fs)
    x_rec = irfft(X)
```

Note the following technical detail: If we plot the amplitude spectrum to analyze the energy distribution across frequencies, we must account for the energy contained in the negative frequencies. Since `rfft()` returns only the non-redundant positive-frequency components, their amplitudes must be adjusted accordingly.

```python
    # x, fs = ...
    X = rfft(x)
    A = np.abs(X) / len(x) # Normalized amplitude
    A[1:-1] *= 2           # Correct for negative freqs.
    phi = np.angle(X)      # Phase (no correction)
```


</span>
</div>

<hr style="max-width:600px; margin:1em 0;">

<!-------------------------------- Question 8 -------------------------------->
<div style="display:flex; max-width:600px; padding:0 0 0.5em 0;">
<b style="min-width:90px;">Question 8:</b>
<span>
What is the purpose of the argument <code>n</code> in <code>fft(x, n)</code>?
</span>
</div>

<div style="display:flex; max-width:600px; padding:0;">
<b style="min-width:90px;">Answer 8:</b>
<span>
The argument <code>n</code> specifies the <strong>length of the DFT</strong>.

If <code>n &gt; len(x)</code>, the signal is <strong>zero-padded</strong>.<br>
If <code>n &lt; len(x)</code>, the signal is <strong>truncated</strong>.<br><br>
This parameter allows control over the length of the transform and the resulting frequency resolution.
</span>
</div>


---

<a id='exercise2'></a>

## **&#9734; Exercise 2 – Spectral analysis of audio data**

In this exercise, we analyze the frequency content of audio data.

### **Instructions**:
1. Load and play an audio file
   * Use the convenience function `load_audio()` for this.
   * Choose a file under ../data/signals, but you can also use your own audio. 
   * Note: The audio file should not be too long (30s max), as the visualization   
     is not optimized for a large number of samples
   * Make sure that the data is a **stereo** signal with two channels: `shape (N, 2)`
2. Visualize the time-domain data for each channel (left and right).
3. Compute the DFT of each channel and visualize the amplitude spectrum.
4. Compute the inverse DFT of each channel and visualize the time-domain data.
5. Play the original audio and the reconstructed audio.

In [ ]:
######################
###    EXCERISE    ###
######################

# Step 1:
# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/fail.mp3")
x, fs = isp.load_audio("../data/signals/boo.mp3")
nchannels, nsamples = x.shape
print("The signal has {} channels and {} samples.".format(nchannels, nsamples))

# Play the signal
display(Audio(x, rate=fs))

# Compute the duration of the signal
duration = ...

# Step 2:
# Visualize the data per channel
...

# Step 3:
# Compute the FFT of the signal
T = 1/fs
...
xf = ...
tf = ...

# Plot the amplitude spectrum of the signal
# Don't forget to scale the amplitude spectrum by 1/N
# (or 2/N for the positive frequency components).
# See the "Technical Note *" in the introduction.
xf_amplitude = ...
...

# Step 4:
# Reconstruct the signal.
x_reconstruct = ...

# Step 5:
# Compare the results
display(Audio(x, rate=fs))
display(Audio(x_reconstruct, rate=fs))

In [ ]:
######################
###    SOLUTION    ###
######################

# Step 1:
# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/fail.mp3")
x, fs = isp.load_audio("../data/signals/boo.mp3")
x, fs = isp.load_audio("../data/signals/yeah.mp3")
nchannels, N = x.shape
print("The signal has {} channels and {} samples.".format(nchannels, N))

# Compute the duration of the signal
duration = N / fs

Ts = 1 / fs                     # Sampling period
t = np.arange(0, duration, Ts)  # Timestamps
assert len(t) == N, "The number of timestamps and samples should match."

# Play the signal
display(Audio(x, rate=fs))

# Step 2:
# Plot the signal per channel
xmax = np.abs(x).max()
fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
axes[0].plot(t, x[0,:], color=isp.PALETTE[0], 
             label="Left channel")
axes[0].set_ylim([-xmax, xmax])
axes[0].set_ylabel("Amplitude")
axes[0].grid(axis="x", alpha=0.5)
axes[0].legend()
axes[1].plot(t, x[1,:], color=isp.PALETTE[1], 
             label="Right channel")
axes[1].set_ylim([-xmax, xmax])
axes[1].set_ylabel("Amplitude")
axes[1].grid(axis="x", alpha=0.5)
axes[1].legend()
axes[2].plot(t, x[1,:]-x[0,:], color=isp.PALETTE[2], 
             label="Difference")
axes[2].set_ylim([-xmax, xmax])
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Amplitude")
axes[2].grid(axis="x", alpha=0.5)
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 3:
# Perform Fourier transform
X = np.fft.rfft(x, N)
fk = np.fft.rfftfreq(N, 1/fs)

# Note how the real-valued FFT has been computed for both channels!
print("The signal has {} channels and {} samples.".format(nchannels, N))
print("The spectrum has {} channels and {} samples.".format(*X.shape))

# Extract amplitude and phase
X_amplitude = np.abs(X)/N       # Normalized amplitude
X_amplitude[1:-1,:] *= 2        # Correct for omitted negative frequencies
X_phase = np.angle(X)           # Phase spectrum (no correction needed)

# Visualize Fourier spectrum
plt.figure(figsize=(9, 6))

# Plot amplitude spectrum
plt.subplot(2, 1, 1)
plt.plot(fk, X_amplitude[0,:], "-")
plt.title("Amplitude Spectrum", fontweight="bold")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.grid(axis="y", alpha=0.5)

plt.subplot(2, 1, 2)
plt.plot(fk, X_amplitude[1,:], "-")
plt.title("Phase Spectrum", fontweight="bold")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Phase (rad)")
plt.grid(False)
plt.grid(axis="y", alpha=0.5)

# plt.xlim([0,5000])
plt.tight_layout()
plt.show()

In [ ]:
# Step 4:
# Reconstruct signal
x_reconstructed = np.fft.irfft(X)

# Step 5:
# Compare the results
print("Original")
display(Audio(x, rate=fs))
print("Reconstructed")
display(Audio(x_reconstructed, rate=fs))

---

<a id='exercise3'></a>

## **&#9734; Exercise 3 – FFT of a periodic signal**

We have seen how to compute the **Fourier series** for a periodic signal. In this exercise, we compute the Fourier transform of such a signal.

### **Instructions**
* Create a periodic signal using [`scipy.signals`](https://docs.scipy.org/doc/scipy/reference/signal.html#waveforms) (e.g. a rectangular wave).
* Compute the FFT for this signal
* Visualize the amplitude and phase spectrum
* Compute the FFT for a delayed version of this signal
* Visualize the amplitude and phase spectrum of the delayed signal
* Compare and discuss

In [ ]:
######################
###    EXCERISE    ###
######################
from scipy.signal import square
f = 1           # Signal frequency
fs = 100        # Sampling frequency --> play with this value!
phase = 0.      # Phase of the signal --> play with this value!
duration = 2.   # Duration of the signal in seconds
t = ...
x = square(...)

# Compute the FFT of the signal
Ts = 1/fs
N = len(t)
xf = ...
tf = ...

# Compute the amplitude spectrum of the signal

In [ ]:
######################
###    SOLUTION    ###
######################

from scipy.signal import square

# Create a rectangular signal using scipy.signal.square

f = 1           # Signal frequency
fs = 100        # Sampling frequency --> play with this value!
phase = 0.      # Phase of the signal --> play with this value!
duty = 0.15
duration = 2

# Create timestamps and signal
t = np.arange(0, duration, 1/fs)
x = square(2 * np.pi * f * t - phase, duty=duty)*0.5 + 0.5

# Fourier transform
N = len(t)
X = np.fft.fft(x)
fk = np.fft.fftfreq(len(x), 1/fs)

# Compute the normalized amplitude and phase of the FFT.
# Note: In this example, we use the full FFT (not rfft), which includes 
#       both the positive and negative frequency components. Hence, we need 
#       to divide by N (not 2/N).
X_amp = 1/N*np.abs(X)
X_phase = np.angle(X)

# Set the phase and amplitude to NaN for small amplitudes
X_phase[X_amp < 1e-7] = np.nan
X_amp[X_amp < 1e-7] = np.nan

# So far, we have compute the DFT of a discrete rectangular pulse signal, 
# which yields a discrete spectrum. The following lines illustrate the 
# alternative approaches we have discussed in the lecture:
# A) Fourier series of a rectangular pulse wave (discrete frequency components)
# B) Fourier transform of a rectangular function (continuous frequency spectrum)

# A) Theoretical Fourier transform of a rectangular pulse (alternative formula)
#    See lecture: SW03 / Fourier series / Example: Rectangular pulse
if phase == 0:
    k = np.arange(-fs/2, fs/2+1)
    # Trick: Avoid warning for division by zero for k=0
    k[k==0] = np.nan  
    # Fourier coefficients, from the formulary
    a0 = duty
    ak = 1/(np.pi*k) * np.sin(2*np.pi*k*duty)
    bk = 2/(np.pi*k) * np.sin(np.pi*k*duty)**2
    # Amplitude of the Fourier series components
    X_theoA_amp = np.sqrt(ak**2 + bk**2)/2
    # Fill in the amplitude for k=0 (the DC component)
    X_theoA_amp[np.isnan(k)] = a0
    k[np.isnan(k)] = 0
    # Frequencies of the Fourier series components
    f_theoA = k*f  

# B) Theoretical Fourier transform of a rectangular pulse.
#    See lecture: SW03 / Fourier transformation / Example: Rectangular function
if phase == 0:
    t_on = duty*1/f
    f_theoB = np.linspace(-fs/2, fs/2, 1000)
    X_theoB = t_on*f * np.sinc(t_on*f_theoB)
    X_theoB_amp = np.abs(X_theoB)


# Visualize the signal and its Fourier transform
fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=False)
axes[0].plot(t,x)
axes[0].set_title("Original signal", fontweight="bold")
axes[0].set_xlabel("t[n] (in s)")
axes[0].set_ylabel("x[n]")
axes[0].grid(axis="y", alpha=0.5)

axes[1].stem(fk, X_amp, label="DFT")
axes[1].set_title("Amplitude spectrum", fontweight="bold")
axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_ylabel("Amplitude")
axes[1].grid(axis="y", alpha=0.5)

axes[2].stem(fk, X_amp, label="DFT")
axes[2].set_title("Amplitude spectrum (detailed view)", fontweight="bold")
axes[2].set_xlabel("Frequency [Hz]")
axes[2].set_ylabel("Amplitude")
axes[2].grid(axis="y", alpha=0.5)
axes[2].set_xlim([-10, 10])

axes[3].stem(fk, X_phase, label="DFT")
axes[3].set_title("Phase spectrum", fontweight="bold")
axes[3].set_xlabel("Frequency [Hz]")
axes[3].set_ylabel("Phase")
axes[3].grid(axis="y", alpha=0.5)

if phase == 0:
    # Plot with a vertical offset to avoid overlap
    offset = 0
    axes[1].plot(f_theoA, X_theoA_amp+offset, 
                 color=isp.PALETTE[2], 
                 label="Fourier series", 
                 marker="x", 
                 markerfacecolor="none") 
    axes[1].plot(f_theoB, X_theoB_amp+2*offset, 
                 color=isp.PALETTE[1], 
                 label="Fourier transform") 
    axes[2].plot(f_theoA, X_theoA_amp+offset, 
                 color=isp.PALETTE[2], 
                 label="Fourier series", 
                 marker="x", 
                 markerfacecolor="none") 
    axes[2].plot(f_theoB, X_theoB_amp+2*offset, 
                 color=isp.PALETTE[1], 
                 label="Fourier transform") 
    # Add legends (outside the plot)
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.05, 1))
    axes[2].legend(loc="upper left", bbox_to_anchor=(1.05, 1))

plt.tight_layout()

### **Observations**
- We can apply the DFT/FFT to mimick FS/CTFT under some situations.
- The numerical output of the FFT/DFT matches the formulas derived in the lecture.
- The parallels between FS, CTFT, and DFT hold only if the finite-length signal used for the DFT is consistent with the periodicity and assumptions of the signals used in the other methods.
- The phase spectrum is strongly influenced by the phase of the input signal, whereas the amplitude spectrum primarily reflects the strength of the frequency components.

---

<a id='exercise4'></a>

## **&#9734; Exercise 4 – Spectrograms**

A spectrogram is a time–frequency representation that shows how the spectral content of a signal changes over time. It is computed by taking the Fourier transform of short, overlapping time windows (the short‑time Fourier transform, STFT) and visualizing the magnitude as a color‑coded image. Each vertical slice represents the spectrum at a specific moment, while the horizontal axis tracks the evolution of frequencies across time. 

Spectrograms are especially useful for analyzing non‑stationary signals, whose frequency characteristics vary—such as speech, music, seismic data, or biomedical signals. They help reveal transient events, repeating patterns, harmonics, and other features that are not visible in the raw time‑domain signal. Because of this, spectrograms are widely used in audio processing, diagnostics, machine learning, and many exploratory signal‑analysis tasks. ([Image source](https://commons.wikimedia.org/wiki/File:Spectrogram-19thC.png))

![Spectrogram](../data/doc/spectrogram.jpg)

In Python, there are several ways to create spectrograms. Two commonly used options are [`ShortTimeFFT.spectrogram()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.ShortTimeFFT.html) from SciPy and [`plt.specgram()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.specgram.html) from Matplotlib. For a practical introduction and examples, see the tutorial linked [here](https://scicoding.com/how-to-do-spectrogram-in-python/).

### **Instructions**
* Load an audio file using `isp.load_audio()``
* Select one channel
* Create spectogram
* Interpret the results


In [ ]:
######################
###    EXERCISE    ###
######################

# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/boo.mp3")
x = x[0, :]
duration = len(x) / fs
duration_max = 1   # Clip the signal to 1 second max
x = x[:int(duration_max*fs)]

# Choose an appropriate window length
# Should be a power of 2.
win = 1024

# Create a spectrogram
...

In [ ]:
######################
###    SOLUTION    ###
######################

# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/boo.mp3")
x, fs = isp.load_audio("../data/signals/fail.mp3")
x = x[0, :]
duration = len(x) / fs
duration_max = 3   # Clip the signal to 1 second max
#x = x[:int(duration_max*fs)]

# Choose an appropriate window length
# Should be a power of 2.
win = 1024
plt.specgram(x, NFFT=1024, Fs=fs, mode='magnitude',)
plt.title("matplotlib: specgram");
plt.show()

# Alternative package that is cabable of plotting: librosa
# We need to install it, if iit is not available.
try:
    import librosa
except ImportError:
    %pip install librosa
    pass
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
S = librosa.feature.melspectrogram(y=x, sr=fs, n_mels=128,
                                    fmax=8000)
S_dB = librosa.power_to_db(S, ref=np.max)
img = librosa.display.specshow(S_dB, x_axis="time",
                               y_axis="mel", sr=fs,
                               n_fft=12000,
                               fmax=12000, ax=ax)
fig.colorbar(img, ax=ax, format="%+2.0f dB")
ax.set(title="librosa: mel-frequency spectrogram");


---

<a id='exercise5'></a>

## **&#9734; Exercise 5 – Naive filtering**

In this last exercise, we want to filter a signal by zeroing the frequencies that we wish to have removed. Does this naive filtering approach work?

### **Instructions**
* Load an audio file using `isp.load_audio()``
* Select one channel
* Compute the FFT of the signal
* Visualize the magnitude spectrum
* Set the highest most frequencies to zero
* Compute the inverse FFT
* Listen to the result

In [ ]:
######################
###    EXERCISE    ###
######################

# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/bees.mp3")
x = x[0, :]
duration = len(x) / fs
duration_max = 10   # Clip the signal to 10 seconds max
x = x[:int(duration_max*fs)]

# Compute the FFT of the signal (use rfft)
xf = ...
tf = ...

# Visualize the amplitude spectrum of the signal
... 

# Zero out the upper 30% of the FFT
xf[:int(0.7*len(xf))] = 0

# Reconstruct the signal
x_reconstructed = ...

# Listen to the result
print("Original")
display(Audio(x, rate=fs))
print("Reconstructed")
display(Audio(x_reconstructed, rate=fs))

In [ ]:
######################
###    SOLUTION    ###
######################

# Load the audio file, return the signal and the sampling frequency
x, fs = isp.load_audio("../data/signals/bees.mp3")
#x, fs = isp.load_audio("../data/signals/yeah.mp3")

# Use only the first channel.
x = x[0, :]
duration = len(x) / fs
duration_max = 10   # Clip the signal to 10 seconds max
x = x[:int(duration_max*fs)]

# Optionally add some noise
# x += np.random.normal(0, 0.1, x.shape)

# Compute the FFT of the signal (use rfft)
xf = np.fft.rfft(x)
tf = np.fft.rfftfreq(len(x), 1/fs)

# Ensure amplitude-preserving FT
# (FFT is energy preserving)
xf_amp = 2/len(x)*np.abs(xf)

# Create ideal filter in the frequency domain
f_critical = 5000
hf = (tf < f_critical)#*2/len(tf)
#hf *= 2/len(tf)
hf_amp = hf

# Visualize the amplitude spectrum of the signal
plt.figure(figsize=(8, 4))
plt.plot(tf, xf_amp, "-", label="Signal")
# Display the ideal filter – scaled by 1/100 for better visualization
plt.plot(tf, hf_amp/100, "-", label="Ideal filter")
plt.legend()

plt.title("Amplitude Spectrum", fontweight="bold")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.grid(axis="y", alpha=0.5)
plt.show()

# Let's apply the filter
yf = xf * hf
y = np.fft.irfft(yf)

# Listen to the result
print("Original")
display(Audio(x, rate=fs))
print("Filtered")
display(Audio(y, rate=fs))

# Visualize the signal
fig, axes = plt.subplots(2, 1, figsize=(8, 4))
axes[0].plot(x, label="Original", color=isp.PALETTE[0])
axes[0].legend()
axes[1].plot(y, label="Filtered", color=isp.PALETTE[1])
axes[1].legend();

Naive filtering works, but it introduces artifacts. This happens because we apply a very abrupt modification in the frequency domain, for example by setting certain frequency components exactly to zero. Such sharp changes correspond to long and oscillatory patterns in the time domain.

When we transform the signal back using the inverse FFT, these oscillations appear as artifacts in the reconstructed signal.

In other words, by strongly modifying the spectrum we also alter the time-domain structure of the signal. In the next week, we will see how to design smoother filters that reduce these artifacts.